# Ⅰ第2回 オプション課題：勾配が消えても学習できるのはなぜか ― Adam と SGD

部屋C では，シグモイドを 6 層重ねると入力側の層1の勾配が出力側よりずっと小さくなった．
「読む」の図のとおりなら，層1の重みはほとんど直されず，学習は進まないはず．
**ところが正解率は大きくは下がらなかった**（部屋C の報告の問 3）．

**問い**：

1. 勾配が何万分の 1 になっても，Adam で学習できるのはなぜか
2. Adam をやめて，勾配に学習率を掛けて引くだけの SGD にすると何が起きるか
3. 勾配消失を，最適化手法に頼らずに直す方法はあるか

**鍵**：「勾配の大きさ」と「重みが実際に動いた量」は別物．この 2 つを層ごとに並べて記録する仕組みを **自分で作る**．

題材は第1回 GW3 試行①（FashionMNIST × MLP，5 エポック）．**3〜4 グループで組み**，8 つの条件を分担して全員分で 1 枚の表とグラフを作る．
1 条件の実行時間の目安は 1〜3 分．TODO は 4 つ．ヒントは授業ページの「オプション課題」にある（まず自分で考える）．

1. 実行する前に，下の「予測」を書く
2. (3) に自分の担当を書き，(4)〜(7) の TODO を埋めて実行する
3. (共有) の手順で CSV を仲間と共有する
4. (9) で全員分の表とグラフを作り，(10) の考察を書く（提出）

## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- シグモイド 6 層を SGD（学習率 0.1）で学習すると，5 エポック後の検証正解率は何%くらいか：＿＿％
- その学習率を 10 倍（1.0）にすると，どうなるか：＿＿
- Adam で学習したとき，入力側の層1の重みは，出力側の層と比べて「よく動く／同じくらい／ほとんど動かない」のどれか：＿＿

## (1) ライブラリのインポートと使用デバイス

In [ ]:
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from common.device import get_device, synchronize
from common.fashion import loaders            # GW3 試行①と同じデータ（訓練 48,000 枚・検証 12,000 枚）
from common.gradnorm import GradNormRecorder  # 部屋A で組み立てたものと同じ（層ごとの勾配の大きさ）
from common.logger import ResultLogger
from common.todo import todo_check
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # Mac の日本語フォント（文字化け防止）

device = get_device()
print("使用デバイス:", device)

## (2) データ（GW3 試行①と同じ）

In [ ]:
torch.manual_seed(0)
train_loader, val_loader = loaders(batch_size=64)
print("1 エポックのステップ数:", len(train_loader))

## (3) 担当の条件

組んだグループどうしで 8 つの条件を分担する（授業ページの分担表）．余裕があれば複数担当する．

| 番号 | 活性化関数 | 隠れ層 | BatchNorm | 最適化手法 | 学習率 | 何と比べるか |
|---|---|---|---|---|---|---|
| 1 | sigmoid | 6 | なし | Adam | 0.001 | 部屋C と同じ．基準 |
| 2 | sigmoid | 6 | なし | SGD | 0.1 | 1 と最適化手法だけが違う |
| 3 | sigmoid | 6 | なし | SGD | 1.0 | 2 の学習率を 10 倍 |
| 4 | sigmoid | 2 | なし | SGD | 0.1 | 2 を浅くする |
| 5 | tanh | 6 | なし | SGD | 0.1 | 2 の活性化関数を替える |
| 6 | relu | 6 | なし | SGD | 0.1 | 2 の活性化関数を替える |
| 7 | relu | 6 | なし | Adam | 0.001 | 6 と最適化手法だけが違う |
| 8 | sigmoid | 6 | あり | SGD | 0.1 | 2 に BatchNorm を足す（TODO 4） |

In [ ]:
CONDITIONS = {
    1: ("sigmoid", 6, False, "adam", 0.001),
    2: ("sigmoid", 6, False, "sgd", 0.1),
    3: ("sigmoid", 6, False, "sgd", 1.0),
    4: ("sigmoid", 2, False, "sgd", 0.1),
    5: ("tanh", 6, False, "sgd", 0.1),
    6: ("relu", 6, False, "sgd", 0.1),
    7: ("relu", 6, False, "adam", 0.001),
    8: ("sigmoid", 6, True, "sgd", 0.1),
}
COND = 2            # ← 分担表で決めた番号（1〜8）に書き換える
if COND not in CONDITIONS:
    raise ValueError("COND は 1〜8 のどれかにする")
ACT, DEPTH, BN, OPT, LR = CONDITIONS[COND]
NAME = f"{ACT}_d{DEPTH}{'_bn' if BN else ''}_{OPT}_lr{LR}"
print(f"条件 {COND}：{NAME}")

## (4) 重みが実際に動いた量を記録する仕組み

部屋A の `GradNormRecorder` は，`loss.backward()` で計算された **勾配の大きさ** を記録した．
ここで作る `UpdateRecorder` は，`optimizer.step()` で **重みが実際に動いた量** を記録する．

層ごとに「動いた量 ÷ 重みそのものの大きさ」（相対的な更新量）を 1 つの数にする：

$$\text{更新量} = \frac{\lVert W_{\text{後}} - W_{\text{前}} \rVert}{\lVert W_{\text{前}} \rVert}$$

例えば 0.01 なら，その 1 ステップで重みが 1% 動いた．使い方は次のとおり．

```python
loss.backward()
grad_rec.record()     # 勾配の大きさ（部屋A と同じ）
upd_rec.before()      # step の前の重みを控える
optimizer.step()
upd_rec.after()       # step の後の重みと比べる
```

In [ ]:
class UpdateRecorder:
    def __init__(self, model):
        # GradNormRecorder と同じく，全結合層（nn.Linear）の重みを入力に近い順に「層1, 層2, …」と呼ぶ
        self.layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
        self.names = [f"層{i + 1}" for i in range(len(self.layers))]
        self.history = {name: [] for name in self.names}
        self.saved = []

    def before(self):
        # TODO 1: 「...」を消して，step の前の重みの「控え」を作る式を書く
        #   注意：layer.weight をそのまま入れると，step で中身が書き換わり，控えにならない．
        #         勾配の計算から切り離した（detach），別の場所への写し（clone）を作る
        self.saved = [... for layer in self.layers]

    def after(self):
        for name, layer, w_before in zip(self.names, self.layers, self.saved):
            # TODO 2: 「...」を消して，上の式の「更新量」を Python の float で求める式を書く
            #   ヒント：テンソルの大きさ（L2 ノルム）は .norm()，1 要素のテンソルを数にするのは .item()
            self.history[name].append(...)

    def to_frame(self):
        return pd.DataFrame(self.history)       # 行 = ステップ，列 = 層

todo_check("TODO 1・2")

# 動作確認：重みを手で 1% だけ動かすと，どの層も 0.01 と出るはず
_m = nn.Sequential(nn.Linear(4, 3), nn.Linear(3, 2))
_r = UpdateRecorder(_m)
_r.before()
with torch.no_grad():
    for _l in _r.layers:
        _l.weight.mul_(1.01)
_r.after()
print(_r.to_frame().round(4))

## (5) モデル：活性化関数・深さ・BatchNorm を変えられる MLP

部屋C の `make_mlp` に，`bn=True` のとき隠れ層ごとに **BatchNorm**（バッチ正規化）を入れる機能を足す．
BatchNorm は，ミニバッチの中で各ユニットの値を平均 0・ばらつき 1 にそろえ直す層．全結合層と活性化関数の **間** に入れる．

In [ ]:
ACTS = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

def make_mlp(act, depth, bn=False, width=256):
    layers = [nn.Flatten()]
    in_features = 28 * 28
    for _ in range(depth):
        layers.append(nn.Linear(in_features, width))
        if bn:
            # TODO 4: 「...」を消して，幅 width の BatchNorm（1 次元の値用：nn.BatchNorm1d）を作って足す
            layers.append(...)
        layers.append(ACTS[act]())
        in_features = width
    layers.append(nn.Linear(in_features, 10))          # 出力層（活性化関数なし）
    return nn.Sequential(*layers)

if BN:
    todo_check("TODO 4")
torch.manual_seed(0)
model = make_mlp(ACT, DEPTH, BN).to(device)
model

## (6) 最適化手法

- **Adam**（部屋A〜C で使ったもの）：過去の勾配から，重みの 1 つ 1 つについて一歩の大きさを自動で調整する
- **SGD**（確率的勾配降下法）：`重み ← 重み − 学習率 × 勾配`．勾配が小さければ，一歩も小さい

In [ ]:
criterion = nn.CrossEntropyLoss()
if OPT == "adam":
    optimizer = optim.Adam(model.parameters(), lr=LR)
else:
    # TODO 3: 「...」を消して，学習率 LR の SGD を作る（Adam の行と同じ形．optim.SGD を使う）
    optimizer = ...
    todo_check("TODO 3")
print(optimizer)

## (7) 学習（5 エポック）

`loss.backward()` と `optimizer.step()` の間で勾配を，`optimizer.step()` の前後で更新量を記録する．

In [ ]:
def validate(model):
    # 検証データの損失と正解率（GW3 の⑪と同じ）
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

EPOCHS = 5
grad_rec = GradNormRecorder(model)
upd_rec = UpdateRecorder(model)
train_losses, val_log = [], []

synchronize(device)
t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        grad_rec.record()       # 勾配の大きさ
        upd_rec.before()        # step の前の重みを控える
        optimizer.step()
        upd_rec.after()         # 重みが実際に動いた量
        train_losses.append(loss.item())
    val_loss, val_acc = validate(model)
    val_log.append((val_loss, val_acc))
    print(f"エポック {epoch + 1}/{EPOCHS}  訓練損失 {sum(train_losses[-len(train_loader):]) / len(train_loader):.4f}"
          f"  検証損失 {val_loss:.4f}  検証正解率 {val_acc * 100:.2f}%")
synchronize(device)
train_sec = time.perf_counter() - t0
grads, upds = grad_rec.to_frame(), upd_rec.to_frame()
print(f"学習時間 {train_sec:.1f} 秒 | 記録したステップ数 {len(grads)}")

## (8) グラフ：層ごとの勾配と更新量

左が勾配の大きさ，右が重みが実際に動いた量．どちらも対数目盛．
表の「÷層1」は，各層の値が入力側の層1の何倍かを表す（学習の最初の 100 ステップの平均）．

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
grads.rolling(50, min_periods=1).mean().plot(ax=axes[0], logy=True)
axes[0].set_title(f"勾配の大きさ（{NAME}）"); axes[0].set_xlabel("ステップ")
upds.rolling(50, min_periods=1).mean().plot(ax=axes[1], logy=True)
axes[1].set_title(f"更新量 ‖ΔW‖/‖W‖（{NAME}）"); axes[1].set_xlabel("ステップ")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

g100, u100 = grads.head(100).mean(), upds.head(100).mean()
summary = pd.DataFrame({"勾配": g100, "勾配 ÷層1": g100 / g100.iloc[0],
                        "更新量": u100, "更新量 ÷層1": u100 / u100.iloc[0]})
summary

## (記録)

In [ ]:
metrics = {"val_loss": val_log[-1][0], "val_acc": val_log[-1][1], "train_sec": train_sec,
           "grad_first": g100.iloc[0], "grad_last": g100.iloc[-1],
           "upd_first": u100.iloc[0], "upd_last": u100.iloc[-1]}
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d2", exercise="opt1", device=device)
logger.log_many(metrics, condition=f"{COND}:{NAME}", seed=0)
print("書き込み先:", logger.path)

## (共有) CSV を組んだ仲間と集める

1. 自分の `results/c1_d2_opt1_<グループ>.csv` を，組んだ仲間全員に送る（Slack・AirDrop など）
2. 受け取った CSV は，すべて `results/d2_opt_shared/` フォルダに入れる（無ければ作る．同じ名前のファイルは末尾に `_2` などを付けて別名にする）
3. もう 1 条件担当するときは，(3) の `COND` を書き換えて，(3) から下をもう一度実行する（同じ CSV に行が増える）

## (9) 全員分の表とグラフ（何回実行してもよい）

「出力側÷層1」は，出力側の層が入力側の層1の何倍か（学習の最初の 100 ステップ）．1 より大きいほど層1が小さい．

In [ ]:
import glob
files = glob.glob(str(ROOT / "results" / "c1_d2_opt1_*.csv")) + glob.glob(str(ROOT / "results" / "d2_opt_shared" / "*.csv"))
df = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
df = df[(df["day"] == "d2") & (df["exercise"] == "opt1")]
wide = df.pivot_table(index=["group_id", "condition"], columns="metric_name", values="metric_value").reset_index()
wide["勾配 出力側÷層1"] = wide["grad_last"] / wide["grad_first"]
wide["更新量 出力側÷層1"] = wide["upd_last"] / wide["upd_first"]
table = wide.groupby("condition")[["val_acc", "勾配 出力側÷層1", "更新量 出力側÷層1", "train_sec"]].mean()
table = table.sort_index(key=lambda s: s.str.split(":").str[0].astype(int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
table["val_acc"].mul(100).plot.barh(ax=axes[0], color="tab:gray")
axes[0].axvline(10, color="red", ls="--", label="当てずっぽう（10%）")
axes[0].set_xlabel("5 エポック後の検証正解率（%）"); axes[0].legend(); axes[0].invert_yaxis()
table[["勾配 出力側÷層1", "更新量 出力側÷層1"]].plot.barh(ax=axes[1], logx=True)
axes[1].axvline(1, color="black", lw=0.8)
axes[1].set_xlabel("出力側 ÷ 層1（対数目盛．1 より右ほど層1が小さい）"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

print(f"集まった条件：{len(table)} / 8　（ファイル {len(files)} 個，実行 {len(wide)} 回）")
table.round(4)

## (9-2) Adam の一歩を手で確かめる

Adam は，勾配 $g$ をそのまま使わず，次のように一歩を決める（$\beta_1 = 0.9,\ \beta_2 = 0.999$）．

$$m \leftarrow \beta_1 m + (1-\beta_1)\,g, \qquad v \leftarrow \beta_2 v + (1-\beta_2)\,g^2, \qquad
W \leftarrow W - \text{lr} \cdot \frac{\hat m}{\sqrt{\hat v} + \varepsilon}$$

（$\hat m, \hat v$ は，最初のうち $m, v$ が 0 に引っ張られるのを補正したもの．）
下のセルでは，勾配の大きさが 1 万倍違う 2 つの重みに，Adam と SGD で 1 歩ずつ進ませる．実行して，動いた量を比べる．

In [ ]:
for name in ["adam", "sgd"]:
    w = torch.zeros(2, requires_grad=True)
    opt = optim.Adam([w], lr=0.001) if name == "adam" else optim.SGD([w], lr=0.1)
    (w * torch.tensor([1.0, 0.0001])).sum().backward()    # 勾配は 1 と 0.0001（1 万倍違う）
    opt.step()
    print(f"{name:4s}  勾配 {w.grad.tolist()}  →  動いた量 {(-w.detach()).tolist()}")

## (10) 考察（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 2〜3 文．表の数値を使う）．

1. 条件 1（Adam）と条件 2（SGD）で，「勾配 出力側÷層1」と「更新量 出力側÷層1」はそれぞれ何倍だったか．2 つの指標の違いから，Adam で層1が学習できた理由を (9-2) の式と結果を使って説明する：＿＿
2. 条件 2 が学習できなかったのは「学習率が小さすぎたから」と言えるか．条件 3 の結果を根拠に答える：＿＿
3. SGD で学習できた条件（4〜6）と，できなかった条件（2・3）の違いを，「読む」の勾配消失の図（シグモイドの傾きは最大 0.25）を使って説明する：＿＿
4. 条件 8（BatchNorm）で，勾配の比と正解率はどう変わったか．BatchNorm が勾配消失を防ぐしくみを，シグモイドの傾きのグラフ（入力が 0 付近で一番大きい）と結び付けて推測する：＿＿
5. Adam を使えば勾配消失は気にしなくてよい，と言えるか．条件 1 と条件 7・8 の正解率を比べて，自分の考えを書く：＿＿